# DQNA — Distributed Quantum Network Assignment

경계 UE를 여러 zone에 복제하여 각 QPU가 독립·병렬 실행,
측정 후 고전 조정(reconciliation)으로 글로벌 할당 확정.

**시나리오**: 6 UE × 3 AP, AP capacity = 2, 경계 UE = {2, 4}

| Zone | AP | UE (로컬) | 경계 UE |
|------|----|-----------|--------|
| 0 | AP0 | UE0, UE1, UE2 | UE2 |
| 1 | AP1 | UE2, UE3, UE4 | UE2, UE4 |
| 2 | AP2 | UE4, UE5 | UE4 |

**핵심 설계**:  
- 품질 → Ry biased initial state (고품질 UE → |1⟩ 편향)  
- 제약 → Grover oracle (feasible 상태에 phase -1)  
- Diffusion → biased state 기준 반사  
- 분리 원칙: 품질과 제약의 간섭 없음

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
from qiskit import transpile
from qiskit_aer import Aer

import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from distributed_assignment import (
    Network, Zone, partition, build_zone_circuit,
    run_zone, run_all_zones, collect_candidates,
    resolve_boundaries, run_pipeline, brute_force,
    analyze_statevector, _quality_angles,
)

print('Import OK')

## 1. 네트워크 정의 & Partition

In [ ]:
quality = np.array([
    [ 9.0, -1.0, -1.0],   # UE0: Zone 0
    [ 7.5, -1.0, -1.0],   # UE1: Zone 0
    [ 4.0,  6.0, -1.0],   # UE2: boundary (Zone 0/1)
    [-1.0,  8.0, -1.0],   # UE3: Zone 1
    [-1.0,  5.0,  7.0],   # UE4: boundary (Zone 1/2)
    [-1.0, -1.0,  8.5],   # UE5: Zone 2
])

net = Network(n_ue=6, n_ap=3, quality=quality, capacity=[2, 2, 2])

zone_defs = [
    {"ap": 0, "ues": [0, 1, 2]},
    {"ap": 1, "ues": [2, 3, 4]},
    {"ap": 2, "ues": [4, 5]},
]
zones = partition(net, zone_defs)

print('Link Quality Matrix (UE x AP):')
print(quality)
print()
for z in zones:
    bnd = [z.ues[i] for i, b in enumerate(z.boundary) if b]
    print(f'Zone {z.zone_id} (AP{z.ap}): UE{z.ues}, q={z.qualities}, boundary={bnd}')

## 2. Zone 회로 구조

각 zone에 대해 Biased Grover 회로를 생성하고 구조를 시각화합니다.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

for idx, z in enumerate(zones):
    qc = build_zone_circuit(z, grover_iter=1, alpha=0.3)
    qc.draw('mpl', fold=60, ax=axes[idx], style='iqp')
    bnd = [z.ues[i] for i, b in enumerate(z.boundary) if b]
    axes[idx].set_title(
        f'Zone {z.zone_id} (AP{z.ap}) — UE{z.ues}, '
        f'{qc.num_qubits} qubits, depth={qc.depth()}, boundary={bnd}',
        fontsize=12, fontweight='bold'
    )

plt.tight_layout()
plt.show()

## 3. Biased Initial State 분석

alpha에 따른 초기 상태 편향 시각화.  
`Ry(θ)` 회전으로 고품질 UE는 |1⟩ (할당) 확률이 높아집니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, z in zip(axes, zones):
    alphas = np.linspace(0, 0.8, 20)
    for i, ue in enumerate(z.ues):
        probs = []
        for a in alphas:
            thetas = _quality_angles(z.qualities, a)
            p1 = np.sin(thetas[i] / 2) ** 2  # P(|1>)
            probs.append(p1)
        style = '--' if z.boundary[i] else '-'
        ax.plot(alphas, probs, style, label=f'UE{ue} (q={z.qualities[i]:.1f})')
    ax.set_xlabel('alpha')
    ax.set_ylabel('P(assigned)')
    ax.set_title(f'Zone {z.zone_id} (AP{z.ap})')
    ax.legend(fontsize=9)
    ax.set_ylim(0.4, 1.0)
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)

plt.suptitle('Biased Initial State: P(|1>) vs alpha', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Statevector 분석 (Oracle 검증)

측정 전 상태벡터를 확인하여 oracle이 feasible 상태를 올바르게 증폭하는지 검증합니다.

In [ ]:
from qiskit.quantum_info import Statevector

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, z in zip(axes, zones):
    qc = build_zone_circuit(z, grover_iter=2, alpha=0.3)
    qc_nm = qc.remove_final_measurements(inplace=False)
    sv = Statevector.from_instruction(qc_nm)
    probs = sv.probabilities_dict()

    n = len(z.ues)
    assign_probs = {}
    for bs, p in probs.items():
        key = bs[-n:]
        assign_probs[key] = assign_probs.get(key, 0) + p

    sorted_items = sorted(assign_probs.items(), key=lambda x: -x[1])
    labels, vals = zip(*sorted_items)
    colors = []
    for bs in labels:
        hw = sum(int(b) for b in bs)
        if 1 <= hw <= z.cap:
            colors.append('#4C72B0')  # feasible
        else:
            colors.append('#C44E52')  # infeasible

    ax.bar(range(len(labels)), vals, color=colors)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([f'|{l}>' for l in labels], rotation=45, fontsize=9)
    ax.set_ylabel('Probability')
    ax.set_title(f'Zone {z.zone_id} (AP{z.ap})', fontweight='bold')

# legend
from matplotlib.patches import Patch
axes[0].legend(
    handles=[Patch(color='#4C72B0', label='Feasible'),
             Patch(color='#C44E52', label='Infeasible')],
    fontsize=9
)
plt.suptitle('Statevector Probabilities (grover_iter=2, alpha=0.3)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 텍스트 출력
for z in zones:
    analyze_statevector(z, grover_iter=2, alpha=0.3, min_assign=1)

## 5. Shot-based 실행 & 측정 히스토그램

In [ ]:
results = run_all_zones(zones, shots=8192, grover_iter=2, alpha=0.3)

for res in results:
    z = res['zone']
    names = [f'UE{u}' for u in z.ues]
    sel = ', '.join(f"{nm}={'O' if s else 'X'}" for nm, s in zip(names, res['selection']))
    print(f"Zone {z.zone_id}: {sel}")
    top5 = sorted(res['counts'].items(), key=lambda x: -x[1])[:5]
    print(f"  Top-5: {dict(top5)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, res in enumerate(results):
    z = res['zone']
    plot_histogram(res['counts'], ax=axes[idx],
                   title=f'Zone {z.zone_id} (AP{z.ap})',
                   bar_labels=False, sort='value_desc')

plt.tight_layout()
plt.show()

## 6. 경계 UE 후보 수집 & 충돌 해소

In [ ]:
cands = collect_candidates(results)

print('UE별 후보:')
for ue in sorted(cands):
    entries = cands[ue]
    bnd = ' <- BOUNDARY' if len(entries) > 1 else ''
    info = ', '.join(
        f"AP{e['ap']}({'O' if e['selected'] else 'X'} q={e['quality']:.1f})"
        for e in entries
    )
    print(f'  UE{ue}: {info}{bnd}')

final = resolve_boundaries(cands, net)
print('\nFinal Assignment:')
for ue in sorted(final):
    ap = final[ue]
    if ap is not None:
        print(f'  UE{ue} -> AP{ap} (q={net.quality[ue, ap]:.1f})')
    else:
        print(f'  UE{ue} -> unassigned')
print(f'\nSingle-round score: {net.score(final):.1f}')

## 7. Multi-Round Pipeline

다중 라운드 실행 + zone별 top-K 조합 탐색으로 최적해를 안정적으로 찾습니다.

In [ ]:
pipe = run_pipeline(
    net, zones,
    shots=4096, grover_iter=2, alpha=0.3,
    min_assign=1, rounds=10, top_k=3, verbose=True
)

print(f"\nBest assignment (score={pipe['score']:.1f}):")
for ue in sorted(pipe['assignment']):
    ap = pipe['assignment'][ue]
    if ap is not None:
        print(f'  UE{ue} -> AP{ap} (q={net.quality[ue, ap]:.1f})')
    else:
        print(f'  UE{ue} -> unassigned')

## 8. Benchmark: Optimal vs DQNA

In [ ]:
opt_assign, opt_score = brute_force(net)
single_score = net.score(final)
multi_score = pipe['score']

print(f"{'Method':<25} {'Score':>8} {'Ratio':>8}")
print(f"{'-'*25} {'-'*8} {'-'*8}")
print(f"{'Classical optimal':<25} {opt_score:8.1f} {'--':>8}")
print(f"{'DQNA (single round)':<25} {single_score:8.1f} {single_score/opt_score*100:7.1f}%")
print(f"{'DQNA (multi-round)':<25} {multi_score:8.1f} {multi_score/opt_score*100:7.1f}%")

fig, ax = plt.subplots(figsize=(7, 4))
methods = ['Classical\nOptimal', 'DQNA\n(single)', 'DQNA\n(multi-round)']
scores = [opt_score, single_score, multi_score]
colors = ['#55A868', '#C44E52', '#4C72B0']
bars = ax.bar(methods, scores, color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', fontweight='bold')
ax.set_ylabel('Total Link Quality')
ax.set_title('Centralized Optimal vs DQNA', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Grover Iteration 수에 따른 성능 변화

feasible/total 비율에 따라 최적 iteration 수가 달라집니다.  
6/8 feasible (3 UE, cap=2)의 경우 sin²(3θ)=0 → iter=1에서 overshooting 발생.

In [ ]:
iter_scores = {gi: [] for gi in range(1, 6)}

for gi in range(1, 6):
    for _ in range(10):
        p = run_pipeline(net, zones, shots=4096, grover_iter=gi,
                         alpha=0.3, min_assign=1,
                         rounds=5, top_k=3, verbose=False)
        iter_scores[gi].append(p['score'])

fig, ax = plt.subplots(figsize=(7, 4))
positions = list(range(1, 6))
avgs = [np.mean(iter_scores[gi]) for gi in positions]
stds = [np.std(iter_scores[gi]) for gi in positions]

ax.errorbar(positions, avgs, yerr=stds, fmt='o-', capsize=5,
            color='#4C72B0', linewidth=2, label='DQNA (multi-round)')
ax.axhline(y=opt_score, color='#55A868', linestyle='--',
           linewidth=2, label=f'Optimal ({opt_score:.1f})')
ax.set_xlabel('Grover Iterations')
ax.set_ylabel('Score')
ax.set_title('DQNA Score vs Grover Iterations', fontweight='bold')
ax.set_xticks(positions)
ax.legend()
plt.tight_layout()
plt.show()

for gi in positions:
    print(f'iter={gi}: avg={np.mean(iter_scores[gi]):.1f} '
          f'std={np.std(iter_scores[gi]):.1f} '
          f'best={max(iter_scores[gi]):.1f}')

## 10. Alpha 감도 분석

biased initial state 강도(alpha)에 따른 성능 변화.

In [ ]:
alpha_range = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7]
alpha_scores = {a: [] for a in alpha_range}

for a in alpha_range:
    for _ in range(10):
        p = run_pipeline(net, zones, shots=4096, grover_iter=2,
                         alpha=a, min_assign=1,
                         rounds=5, top_k=3, verbose=False)
        alpha_scores[a].append(p['score'])

fig, ax = plt.subplots(figsize=(7, 4))
avgs = [np.mean(alpha_scores[a]) for a in alpha_range]
stds = [np.std(alpha_scores[a]) for a in alpha_range]

ax.errorbar(alpha_range, avgs, yerr=stds, fmt='s-', capsize=5,
            color='#DD8452', linewidth=2)
ax.axhline(y=opt_score, color='#55A868', linestyle='--',
           linewidth=2, label=f'Optimal ({opt_score:.1f})')
ax.set_xlabel('Alpha (bias strength)')
ax.set_ylabel('Score')
ax.set_title('DQNA Score vs Alpha', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

for a in alpha_range:
    print(f'alpha={a:.1f}: avg={np.mean(alpha_scores[a]):.1f} '
          f'std={np.std(alpha_scores[a]):.1f}')

## 11. Transpile 후 게이트 카운트

In [ ]:
backend = Aer.get_backend('aer_simulator')

for z in zones:
    qc = build_zone_circuit(z, grover_iter=2, alpha=0.3)
    qc_t = transpile(qc, backend)
    ops = dict(qc_t.count_ops())
    print(f'Zone {z.zone_id} (AP{z.ap}):')
    print(f'  Original: {qc.num_qubits} qubits, depth={qc.depth()}')
    print(f'  Transpiled: depth={qc_t.depth()}, gates={ops}')

## 12. 네트워크 토폴로지 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# AP 위치
ap_pos = {0: (1, 3), 1: (4, 3), 2: (7, 3)}
# UE 위치
ue_pos = {
    0: (0.3, 1), 1: (1.7, 1),
    2: (2.5, 1.5),
    3: (4, 1),
    4: (5.5, 1.5),
    5: (7, 1),
}

# Zone 배경
zone_colors = ['#E8F0FE', '#FEF0E8', '#E8FEF0']
zone_ranges = [(0, 3.2), (2.2, 6.0), (5.0, 8.0)]
for zi, (x0, x1) in enumerate(zone_ranges):
    ax.axvspan(x0, x1, alpha=0.3, color=zone_colors[zi],
               label=f'Zone {zi} (AP{zi})')

# AP
for ap, (x, y) in ap_pos.items():
    ax.plot(x, y, 's', markersize=20, color='#C44E52', zorder=5)
    ax.text(x, y + 0.3, f'AP{ap}', ha='center', fontweight='bold', fontsize=11)

# UE + 링크
boundary = {2, 4}
for ue, (x, y) in ue_pos.items():
    marker_color = '#DD8452' if ue in boundary else '#4C72B0'
    ax.plot(x, y, 'o', markersize=15, color=marker_color, zorder=5)
    ax.text(x, y - 0.4, f'UE{ue}', ha='center', fontsize=10)

    # 링크
    for ap in range(3):
        if quality[ue, ap] > 0:
            ax.plot([x, ap_pos[ap][0]], [y, ap_pos[ap][1]],
                    '-', color='gray', alpha=0.4, linewidth=1)
            mx = (x + ap_pos[ap][0]) / 2
            my = (y + ap_pos[ap][1]) / 2
            ax.text(mx, my, f'{quality[ue, ap]:.0f}',
                    fontsize=8, ha='center', color='gray')

# 최적 할당 표시
for ue, ap in opt_assign.items():
    if ap is not None:
        ux, uy = ue_pos[ue]
        ax.plot([ux, ap_pos[ap][0]], [uy, ap_pos[ap][1]],
                '-', color='#55A868', linewidth=2.5, alpha=0.8)

ax.set_xlim(-0.5, 8.5)
ax.set_ylim(0, 4.5)
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=9)
ax.set_title('Network Topology & Optimal Assignment (green links)', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()